# Анализ для игры "Секреты темнолесья".
## Проект спринта № 7.

- Автор: Козлачкова Светлана
- Дата: 09.11.2025

### Цели и задачи проекта

<font color='#777778'>В данном проекте мы используем датасет `datasets/new_games.csv`, который содержит информацию о продажах игр разных жанров и платформ, а так же пользовательские и экспертные оценки игр. Мы познакомимся с данными датасета, проверим их на корректность, проведем предобработку данных, и получим необходимый срез. Так же мы категоризируем оценки пользователей и экспертов, и выделим топ-7 платформ по количеству игр, выпущенных за рассматриваемый период.</font>

### Описание данных

<font color='#777778'> Данные `/datasets/new_games.csv` содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:
- `Name` -  название игры.
- `Platform` - название платформы.
- `Year of Release` - год выпуска игры.
- `Genre` - жанр игры.
- `NA sales` - продажи в Северной Америке (в миллионах проданных копий).
- `EU sales` - продажи в Европе (в миллионах проданных копий).
- `JP sales` - продажи в Японии (в миллионах проданных копий).
- `Other sales` - продажи в других странах (в миллионах проданных копий).
- `Critic Score` - оценка критиков (от 0 до 100).
- `User Score` - оценка пользователей (от 0 до 10).
- `Rating` - рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.
</font>

### Содержимое проекта

<font color='#777778'>
    
1. [Загрузка и знакомство с данными](#load)
2. [Проверка ошибок в данных и их предобработка](#check)
3. [Фильтрация данных](#filter)
4. [Категоризация данных](#category)
5. [Итоговый вывод](#itog)</font>


## 1. <a id='load'>Загрузка и знакомство с данными.</a>

<font color='#777778'> Загрузим необходимые библиотеки для анализа данных и данные из датасета `/datasets/new_games.csv`. Затем выведем основную информацию о данных с помощью метода `info()` и первые строки датафрейма.</font>

In [1]:
# импортируем необходимые билиотеки
import pandas as pd
import numpy as np

In [2]:
# натроим отображение строк
pd.options.display.max_rows = 400

In [3]:
# выгружаем данные из датасета datasets/new_games.csv в датафрейм new_games
try:   
    new_games = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')
except FileNotFoundError:
    new_games = pd.read_csv('/datasets/new_games.csv')

In [4]:
# выводим информацию о датафрейме
new_games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [5]:
# Сохраним первоначальные размеры датафрейма
size_before = new_games.shape
# выведем первые 5 строк датафрейма
display(new_games.head())

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


<font color='#777778'> Датасет `new_games.csv` содержит 11 столбцов и 16956 строк, в которых предоставлена информация о продажах игр в разных странах c 2000 по 2013 гг включительно.

<font color='#777778'>    Изучим типы данных и их корректность:

<font color='#777778'>  - **Числовые значения с плавающей запятой (float64)** - четыре столбца представлены типом `float64`. Это `Year of Release`, `NA sales`, `Other sales` и `Critic Score`. Для столбца `Year of Release`, который содержит год выпуска игры, тип данных подобран некорректно. Его лучше преобразовать в тип `int16`, что позволит сэкономить память и повысить скорость работы. Cтолбец `Critic Score` хранит в себе оценки критиков от 0 до 100 и его так же можно преобразовать к типу `int8`.

<font color='#777778'> - **Cтроковые данные (object)** - семь столбцов представлены типом `object`:
Столбцы `Name`, `Platform `, `Genre` хранят строковую информацию о названии игры, платформе и жанре. Для этих столбцов тип данных подобран верно.
Столбец `Rating` так же хранит в себе текстовые данные, но его можно рассматривать как категориальный признак. В этом случае можно использовать тип `category`, чтобы улучшить производительность и оптимизировать память, если набор значений ограничен и заведомо известен.
Столбцы `EU sales`, `JP sales` содержат в себе количество проданных копий (в млн. ед.) и относятся к числовым значениям.Чтобы избежать трудностей в дальнейшем анализе необходимо преобразовать эти столбцы к типу данных `float64`
Столбец `User Score` отражает оценку пользователей от 1 до 10 и тоже относится к числовым значения. Судя по первоначальным данным столбец должен быть представлен в числовом значении с плавающей запятой. Его так же необходимо переобразовать в тип данных `float64`.

<font color='#777778'>После анализа типов данных, видно что для дальнейшей работы как минимум 4 столбца (`Year of Release`, `EU sales`, `JP sales`, `User Score`) необходимо преобразовать к другим типам.</font>

<font color='#777778'>Пропуски встречаются в 6 столбцах: `Name`, `Year of Release`, `Genre`, `Critic Score`, `User Score` и `Rating`.</font>

<font color='#777777'>Названия столбцов датафрейма корректно отражают содержимое данных, но прописаны в неудобном формате.
Для удобства работы в дальнейшем приведем все названия столбцов к единому стилю. 
`snake_case`</font>

---
## 2. <a id='check'>Проверка ошибок в данных и их предобработка</a>
### 2.1. Названия, или метки столбцов датафрейма

<font color='#777778'> Единообразные названия облегчают понимание и интерпретацию данных, что особенно важно при работе с большими объемами информации. Выведем на экран названия всех столбцов датафрейма и проверим их стиль написания.</font>

In [6]:
# Выведем названия столбцов из датафрейма
new_games.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

<font color='#777778'>Мы видим, что названия столбцов отличаются по стилистическому написанию. Для упрощения дальнейшей работы приведем их к стилю `snake_case`.</font>

In [7]:
# Приведем названия столбцов к стилю snake case
new_games.columns = new_games.columns.str.lower().str.replace(" ", "_")
new_games.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

### 2.2. Типы данных

<font color='#777778'>Выбор правильного типа данных критически важен для обеспечения корректности, эффективности и надежности работы с данными.
Не для всех столбцов был правильно определен тип данных при загрузке датафрейма. Это могло случиться из-за наличия смешанных типов данных в некоторых из них, пропусков или `NaN` значений.
Мы планируем преобразовать колонки  `year_of_release` и `critic_score`  к типу `int16` и `int8` соответственно, колонки `eu_sales`, `jp_sales` и `user_score` к типу `float64`, что позволит в дальнейшем не только без проблем проводить различные математические операции с данными, но и оптимизировать память датафрейма.
</font>

<font color='#777778'> Столбцы `year_of_release` и `critic_score` представлены типом `float64` и имеют пропущенные значения. Столбец `year_of_release` содержит год выхода игры и его необходимо привести к типу `Int16`. Столбец `critic_score` содержит оценку критиков от 1 до 100 и уместится в тип данных `Int8`. Приведем эти два столбца к типу данных `Nullable Integer`, который позволяет хранить целые числа с пропущенными значениями `NaN`.</font>

In [8]:
# Сохраним в переменной размер памяти, занимаемый датафреймом до преобразования типов
memory_before = int(new_games.memory_usage().sum())

In [9]:
# Проверим данные в столбце year_of_release
new_games['year_of_release'].unique()

array([2006., 1985., 2008., 2009., 1996., 1989., 1984., 2005., 1999.,
       2007., 2010., 2013., 2004., 1990., 1988., 2002., 2001., 2011.,
       1998., 2015., 2012., 2014., 1992., 1997., 1993., 1994., 1982.,
       2016., 2003., 1986., 2000.,   nan, 1995., 1991., 1981., 1987.,
       1980., 1983.])

In [10]:
# Преобразуем тип данных в колонке year_of_release используя Int16
new_games['year_of_release'] = new_games['year_of_release'].astype('Int16')

In [11]:
# Проверим данные в столбце critic_score
new_games['critic_score'].unique()

array([76., nan, 82., 80., 89., 58., 87., 91., 61., 97., 95., 77., 88.,
       83., 94., 93., 85., 86., 98., 96., 90., 84., 73., 74., 78., 92.,
       71., 72., 68., 62., 49., 67., 81., 66., 56., 79., 70., 59., 64.,
       75., 60., 63., 69., 50., 25., 42., 44., 55., 48., 57., 29., 47.,
       65., 54., 20., 53., 37., 38., 33., 52., 30., 32., 43., 45., 51.,
       40., 46., 39., 34., 35., 41., 36., 28., 31., 27., 26., 19., 23.,
       24., 21., 17., 22., 13.])

<font color='#777778'>Колонка `critic_score` хранит данные по оценке критиков и должна хранить числовые значения от 0 до 100.
Как показывают выгруженные уникальные значения, аномалий в данном столбце нет. </font>

In [12]:
# Преобразуем тип данных в колонке critic_score используя Int8
new_games['critic_score'] = new_games['critic_score'].astype('Int8')

In [13]:
# Проверим данные в столбце eu_sales
new_games['eu_sales'].unique()

array(['28.96', '3.58', '12.76', '10.93', '8.89', '2.26', '9.14', '9.18',
       '6.94', '0.63', '10.95', '7.47', '6.18', '8.03', '4.89', '8.49',
       '9.09', '0.4', '3.75', '9.2', '4.46', '2.71', '3.44', '5.14',
       '5.49', '3.9', '5.35', '3.17', '5.09', '4.24', '5.04', '5.86',
       '3.68', '4.19', '5.73', '3.59', '4.51', '2.55', '4.02', '4.37',
       '6.31', '3.45', '2.81', '2.85', '3.49', '0.01', '3.35', '2.04',
       '3.07', '3.87', '3.0', '4.82', '3.64', '2.15', '3.69', '2.65',
       '2.56', '3.11', '3.14', '1.94', '1.95', '2.47', '2.28', '3.42',
       '3.63', '2.36', '1.71', '1.85', '2.79', '1.24', '6.12', '1.53',
       '3.47', '2.24', '5.01', '2.01', '1.72', '2.07', '6.42', '3.86',
       '0.45', '3.48', '1.89', '5.75', '2.17', '1.37', '2.35', '1.18',
       '2.11', '1.88', '2.83', '2.99', '2.89', '3.27', '2.22', '2.14',
       '1.45', '1.75', '1.04', '1.77', '3.02', '2.75', '2.16', '1.9',
       '2.59', '2.2', '4.3', '0.93', '2.53', '2.52', '1.79', '1.3', '2.6',
   

In [14]:
# Проверим данные в столбце jp_sales
new_games['jp_sales'].unique()

array(['3.77', '6.81', '3.79', '3.28', '10.22', '4.22', '6.5', '2.93',
       '4.7', '0.28', '1.93', '4.13', '7.2', '3.6', '0.24', '2.53',
       '0.98', '0.41', '3.54', '4.16', '6.04', '4.18', '3.84', '0.06',
       '0.47', '5.38', '5.32', '5.65', '1.87', '0.13', '3.12', '0.36',
       '0.11', '4.35', '0.65', '0.07', '0.08', '0.49', '0.3', '2.66',
       '2.69', '0.48', '0.38', '5.33', '1.91', '3.96', '3.1', '1.1',
       '1.2', '0.14', '2.54', '2.14', '0.81', '2.12', '0.44', '3.15',
       '1.25', '0.04', '0.0', '2.47', '2.23', '1.69', '0.01', '3.0',
       '0.02', '4.39', '1.98', '0.1', '3.81', '0.05', '2.49', '1.58',
       '3.14', '2.73', '0.66', '0.22', '3.63', '1.45', '1.31', '2.43',
       '0.7', '0.35', '1.4', '0.6', '2.26', '1.42', '1.28', '1.39',
       '0.87', '0.17', '0.94', '0.19', '0.21', '1.6', '0.16', '1.03',
       '0.25', '2.06', '1.49', '1.29', '0.09', '2.87', '0.03', '0.78',
       '0.83', '2.33', '2.02', '1.36', '1.81', '1.97', '0.91', '0.99',
       '0.95', '2.0'

In [15]:
# Проверим данные в столбце user_score
new_games['user_score'].unique()

array(['8', nan, '8.3', '8.5', '6.6', '8.4', '8.6', '7.7', '6.3', '7.4',
       '8.2', '9', '7.9', '8.1', '8.7', '7.1', '3.4', '5.3', '4.8', '3.2',
       '8.9', '6.4', '7.8', '7.5', '2.6', '7.2', '9.2', '7', '7.3', '4.3',
       '7.6', '5.7', '5', '9.1', '6.5', 'tbd', '8.8', '6.9', '9.4', '6.8',
       '6.1', '6.7', '5.4', '4', '4.9', '4.5', '9.3', '6.2', '4.2', '6',
       '3.7', '4.1', '5.8', '5.6', '5.5', '4.4', '4.6', '5.9', '3.9',
       '3.1', '2.9', '5.2', '3.3', '4.7', '5.1', '3.5', '2.5', '1.9', '3',
       '2.7', '2.2', '2', '9.5', '2.1', '3.6', '2.8', '1.8', '3.8', '0',
       '1.6', '9.6', '2.4', '1.7', '1.1', '0.3', '1.5', '0.7', '1.2',
       '2.3', '0.5', '1.3', '0.2', '0.6', '1.4', '0.9', '1', '9.7'],
      dtype=object)

<font color='#777778'>Столбцы `eu_sales`, `jp_sales` и `user_score` хранят в себе данные по продажам в разных регионах и данные по оценке пользователей от 0 до 10. Данные столбцы имеют тип `object` и содержат в себе строковый тип данных. Приведем их к типу `float64` используя функцию `to_numeric` с аргументом `coerce`, который заменяет все данные, которые не удалось преобразовать в числа на `NaN`.</font>

In [16]:
# Приводим типы данных столбцов eu_sales, jp_sales,user_score к типу float
for column in ['eu_sales', 'jp_sales','user_score']:
    new_games[column] = pd.to_numeric(new_games[column], errors = 'coerce')

In [17]:
# Проверим результат преобразования
new_games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  Int16  
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   Int8   
 9   user_score       7688 non-null   float64
 10  rating           10085 non-null  object 
dtypes: Int16(1), Int8(1), float64(5), object(4)
memory usage: 1.2+ MB


In [18]:
# Проверим какой размер памяти нам удалось сэкономить после преобразования типов данных
memory_after = int(new_games.memory_usage().sum())
print(f'После преобразования столбцов датафрейма удалось сэкономить {round((memory_before - memory_after)/1024,2)} Кб')

После преобразования столбцов датафрейма удалось сэкономить 182.14 Кб


### 2.3. Наличие пропусков в данных

<font color='#777778'>Наличие пропусков в данных может существенно влиять на выводы анализа, и важно учитывать их при интерпретации результатов. Для того, чтобы анализ был полным и точным нам нужно найти пропуски и решить как их обработать. Для начала выведем данные о количестве пропуской и их процентном соотношении. </font>

In [19]:
# Выводим количество пропущенных строк в датафрейме
new_games.isna().sum()

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

In [20]:
# Подсчитываем процент строк с пропусками
pd.DataFrame(round(new_games.isna().mean()* 100,2)).style.background_gradient('coolwarm')

,0
name,0.010000
platform,0.000000
year_of_release,1.620000
genre,0.010000
na_sales,0.000000
eu_sales,0.040000
jp_sales,0.020000
other_sales,0.000000
critic_score,51.390000
user_score,54.660000


<font color='#777778'>В колонках `name`, `year_of_release` и `genre` имеется несущественная доля пропусков. В тоже время в колонках  `critic_score`, `user_scoree` и `rating` имеется значительная доля пропусков (от 40 до 55%).</font>


<font color='#777778'>В столбцах `name` и `genre` всего по два пропуска. Возможно эти пропуски принадлежат одним и тем же играм. Для проверки гипотезы отдельно выгрузим
все строки с пропусками в столбце `name` и `genre`.</font>

In [21]:
# Выведем строки с пропущенными значениями в столбце name
new_games[new_games['name'].isna()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
661,NaN,GEN,1993,NaN,1.78,0.53,0.00,0.08,<NA>,NaN,NaN
14439,NaN,GEN,1993,NaN,0.00,0.00,0.03,0.00,<NA>,NaN,NaN


In [22]:
# Выведем все строки с пропусками в столбце genre
new_games[new_games['genre'].isna()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
661,NaN,GEN,1993,NaN,1.78,0.53,0.00,0.08,<NA>,NaN,NaN
14439,NaN,GEN,1993,NaN,0.00,0.00,0.03,0.00,<NA>,NaN,NaN


<font color='#777778'> Наша гипотеза подтвердилась. Пропуски значений в столбцах `name` и `genre` принадлежат одним и тем же играм с индексами 661 и 14439. Так как эти игры 1993 года выпуска и не нужны нам для дальнейшего анализа их можно удалить.</font>

In [23]:
# Удаляем пропуски в столбцах name и genre
new_games = new_games.dropna(subset = ['name'])

<font color='#777778'> Столбец `year_of_release` содержит 275 строк с пропусками. В процентом соотношении пропуски составляют 1,62% от общего количества строк, и отсутствие этих данных сильно не повлияет на результаты исследования. Так как в дальнейшем нам нужен будет нужен срез по годам, эти строки все равно не попадут в данные и их можно удалить.</font>

In [24]:
# Удалим данные с пропусками в колонке year_of_release
new_games = new_games.dropna(subset = ['year_of_release'])

<font color='#777778'> В дальнейшем мы больше не будем удалять пропуски в столбцах, так как они могут повлиять на результат анализа. 
Поэтому после удаления пропусков в столбцах `name`, `genre` и `year_of_release` снова выведем информацию о пропусках в количественном и процентном соотношении.</font>

In [25]:
# Выводим количество пропущенных строк в датафрейме
new_games.isna().sum()

name                  0
platform              0
year_of_release       0
genre                 0
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8594
user_score         9121
rating             6778
dtype: int64

In [26]:
# Подсчитываем процент строк с пропусками
#round(new_games.isna().sum() / len(new_games) * 100, 2)
pd.DataFrame(round(new_games.isna().mean()* 100,2)).style.background_gradient('coolwarm')

,0
name,0.000000
platform,0.000000
year_of_release,0.000000
genre,0.000000
na_sales,0.000000
eu_sales,0.040000
jp_sales,0.020000
other_sales,0.000000
critic_score,51.530000
user_score,54.690000


<font color='#777778'> Так же в столбцах `eu_sales` `jp_sales` есть небольшое число пропусков. Мы можем заменить их на среднее значение в зависимости от
названия платформы и года выхода игры.</font>

In [27]:
# Заменим пропуски в столбце eu_sales на среднее значение
def fill_eu_sales(row):
    if pd.isna(row['eu_sales']):
        df = new_games[(new_games['platform'] == row['platform']) & (new_games['year_of_release'] == row['year_of_release'])]
        return df['eu_sales'].mean()
    else:
        return row['eu_sales']
new_games['eu_sales'] = new_games.apply(fill_eu_sales, axis = 1)

In [28]:
# Заменим пропуски в столбце jp_sales на среднее значение
def fill_jp_sales(row):
    if pd.isna(row['jp_sales']):
        df = new_games[(new_games['platform'] == row['platform']) & (new_games['year_of_release'] == row['year_of_release'])]
        return df['jp_sales'].mean()
    else:
        return row['jp_sales']
new_games['jp_sales'] = new_games.apply(fill_jp_sales, axis = 1)

<font color='#777778'>В колонка `critic_score` и `user_score` имеет значительное количество пропусков, которые составляют 51,53% и 54,69% соответсвенно. Такая большая доля пропусков может быть вызвана отсутвием рецензий на некоторые игры для определенных платформ, либо сбором данных из различных источников, в которых отсутстовал данный показатель. 
Мы не будем удалять пропущенные строки, так как это может сказаться на других данных. Поэтому мы заменим значения `NaN` на значение-индикатор, которое не может быть использовано в данных.</font>

In [29]:
# Заменим пропущенные значения в колонке critic_score на -1.
new_games['critic_score'] = new_games['critic_score'].replace(np.nan, -1)

In [30]:
# Заменим пропущенные значения в колонке user_score на -1.
new_games['user_score'] = new_games['user_score'].replace(np.nan, -1)

<font color='#777778'>В колонке `rating` содержится 6778 пропущенных значений  (40,64% от общего количества). Данные пропуски могли возникнуть из-за политики рейтинговых агенств. Некоторые игры могут быть исключены из рейтингов по различным причинам, например, если они были отозваны или не получили официального рейтинга от ESRB. Заменим пропуски на значение `NO DATA`</font>

In [31]:
# Заменим пропущенные значения в колонке rating на NO DATA
new_games['rating'] = new_games['rating'].replace(np.nan, 'NO DATA')

### 2.4. Явные и неявные дубликаты в данных

<font color='#777778'>Дубликаты в данных — это повторяющиеся записи, которые могут возникать по разным причинам, таким как ошибки ввода 
информации или технические сбои при сборе и обработке данных. Важно обнаруживать дубликаты, поскольку они могут отрицательно 
влиять на анализ.</font>

<font color='#777778'> Неявные дубликаты представляют собой строки, которые имеют сходство, но не совпадают по всем критериям. 
Неявные дубликаты могут встречаться как в нескольких столбцах, так и в одном, когда разные записи на самом деле обозначают одно и то же.
Такие дублирующие значения могут содержаться в столбцах `name` `platform` `genre` и `rating`.</font>

In [32]:
# Приведем названия игр в столбце `name` к единому написанию
new_games['name'] = new_games['name'].str.lower().str.strip()

In [33]:
# Проверим наличие неявных дубликатов в колонке platform
new_games['platform'].unique()

array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

<font color='#777778'>Хотя в названиях отсутствуют неявные дубликаты, приведем их к единому стилистическому написанию.</font>

In [34]:
# Приведем названия платформ к единому стилю состоящему из заглавных букв
new_games['platform'] = new_games['platform'].str.upper()

<font color='#777778'> Далее проверим уникальные значения, которые находятся в столбце `genre`.</font>

In [35]:
# Проверяем данные в genre
new_games['genre'].unique()

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION', 'SHOOTER',
       'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE', 'SIMULATION',
       'PUZZLE', 'STRATEGY'], dtype=object)

<font color='#777778'>Мы видим, что значения в столбце `genre` дублируются из-за разного стиля написания (строчных и прописных букв). Приведем значения к единому стилю.</font>

In [36]:
# Приводим названия в столбце genre к единому стилю
new_games['genre'] = new_games['genre'].str.lower()

<font color='#777778'> Проверим данные в столбце `rating` на отсутствие неявных дубликатов.</font>

In [37]:
new_games['rating'].unique()

array(['E', 'NO DATA', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP'],
      dtype=object)

<font color='#777778'>В данных встретился рейтинги **'K-A'**, который в данный момент не является категорией рейтинга ESRB. 
**'K-A'** это старый рейтинг, означающий "Kids to Adults" (от детей до взрослых), который был введён в 1994 году и в 1998 году заменён на современный рейтинг **E**(Everyone).</font>

In [38]:
# Заменим рейтинг 'K-A' на 'E'
new_games['rating'] = new_games['rating'].replace('K-A', 'E')

<font color='#777778'>Мы закончили обрабатывать неявные дубликаты, и перейдем к удалению явных дубликатов.</font>

In [39]:
# посмотрим количество явных дубликатов в датасете
int(new_games.duplicated().sum())

235

In [40]:
# удалим дубликаты из датасета
new_games= new_games.drop_duplicates()

In [41]:
# Проверим количество неявных дубликатов
new_games.duplicated(subset = ['name', 'platform', 'year_of_release', 'genre']).sum()

np.int64(1)

<font color='#777778'>Был найден 1 неявный дубликат по столбцам `name`, `platform`, `year_of_release` и `genre`.
Выведем его, для того чтобы понять какую строку из дублирующих можно удалить.</font>

In [42]:
new_games[new_games.duplicated(subset = ['name', 'platform', 'year_of_release', 'genre'], keep=False)]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
606,madden nfl 13,PS3,2012,sports,2.11,0.22,0.0,0.23,83,5.5,E
16465,madden nfl 13,PS3,2012,sports,0.00,0.01,0.0,0.00,83,5.5,E


<font color='#777778'>Удалим строку с номером **16465**, так как у нее отсутствуют данные по продажам в различных регионах.</font>

In [43]:
# Удаляем строку 16465, с помощью метода drop_duplicates и параметра keep = 'last', который помечает дубликатами все строки кроме первой
new_games.drop_duplicates(subset = ['name', 'platform', 'year_of_release', 'genre'], keep='last', inplace = True)

In [44]:
# Проверим изменения
new_games.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16443 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16443 non-null  object 
 1   platform         16443 non-null  object 
 2   year_of_release  16443 non-null  Int16  
 3   genre            16443 non-null  object 
 4   na_sales         16443 non-null  float64
 5   eu_sales         16443 non-null  float64
 6   jp_sales         16443 non-null  float64
 7   other_sales      16443 non-null  float64
 8   critic_score     16443 non-null  Int8   
 9   user_score       16443 non-null  float64
 10  rating           16443 non-null  object 
dtypes: Int16(1), Int8(1), float64(5), object(4)
memory usage: 1.3+ MB


In [45]:
# Проверим как изменились размеры датафрейма после предобработки
size_fin = new_games.shape
print(f'Датафрейм new_games сократился на {size_before[0] - size_fin[0]} строк и на {size_before[1] - size_fin[1]} столбцов')
print(f'Датафрейм new_games сократился на {round(abs((size_fin[0] /size_before[0] - 1)*100),2)} % строк и на {(size_fin[1]/size_before[1] - 1)*100} % столбцов')

Датафрейм new_games сократился на 513 строк и на 0 столбцов
Датафрейм new_games сократился на 3.03 % строк и на 0.0 % столбцов


In [46]:
# Проверим как поменялся размер данных датафрейма после предобработки
memory_fin = int(new_games.memory_usage().sum())
print(f'После преобразования столбцов датафрейма удалось сэкономить {round((memory_before - memory_fin)/1024,2)} Кб')

После преобразования столбцов датафрейма удалось сэкономить 92.39 Кб


#### **Краткие итоги по предобработке данных.**

<font color='#777778'>**Преобразование типов.** В ходе предобработки данных мы произвели преобразование типов данных столбца `year_of_release` к `Int16`;
`critic_score` к `Int8`; `eu_sales`, `jp_sales` и `user_score` к типу `float64`.Данные преобразования позволили сократить размер памяти, занимаемый датафреймом на 182,14 Кб.</font>

<font color='#777778'>**Поиск  и обработка пропущенных значений.** Пропущенные значения встретились в 8 столбцах. 
Пропуски в столбцах `name`, `genre` и `year_of_release` в процентном соотношении составили менее 5% и были удалены из датафрейма.
Пропуски в столбцах `eu_sales` и `jp_sales` были заменены средними значениями в зависимости от названия платформы и года выхода игры.
Пропуски в столбцах `critic_score`, `user_score` и `rating` были заменены на на значения-индикатор.</font>

<font color='#777778'>**Работа с неявными дубликтами.** Неявные дубликаты были найдены и заменены в колонках `genre` и `rating`. А так же были приведены к единому стилистическому написанию колонки `name` и `platform`.</font>

<font color='#777778'>**Удаление явных дубликатов.** Было найдено 235 явных дубликатов и произведена очистка _и 1 неявный дубликат_.
В итоге получился датафрейм, состоящий из _16443_ строк и 11 столбцов и занимаемой памятью меньше, чем первоначальный на 92,3 Кб.
</font>

---

## 3. <a id='filter'>Фильтрация данных</a>

<font color='#777778'>Коллеги из `Секреты темнолесья` хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно.
Отберем данные по этому показателю.</font>

In [47]:
# Сохраним новый срез данных в отдельный датафрейм
df_actual = new_games[(new_games['year_of_release'] >= 2000) & (new_games['year_of_release'] <= 2013)].copy()

In [48]:
# Выведем первые 10 строк нового датафрейма
df_actual.head(10)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,wii sports,WII,2006,sports,41.36,28.96,3.77,8.45,76,8.0,E
2,mario kart wii,WII,2008,racing,15.68,12.76,3.79,3.29,82,8.3,E
3,wii sports resort,WII,2009,sports,15.61,10.93,3.28,2.95,80,8.0,E
6,new super mario bros.,DS,2006,platform,11.28,9.14,6.50,2.88,89,8.5,E
7,wii play,WII,2006,misc,13.96,9.18,2.93,2.84,58,6.6,E
8,new super mario bros. wii,WII,2009,platform,14.44,6.94,4.70,2.24,87,8.4,E
10,nintendogs,DS,2005,simulation,9.05,10.95,1.93,2.74,-1,-1.0,NO DATA
11,mario kart ds,DS,2005,racing,9.71,7.47,4.13,1.90,91,8.6,E
13,wii fit,WII,2007,sports,8.92,8.03,3.60,2.15,80,7.7,E
14,kinect adventures!,X360,2010,misc,15.00,4.89,0.24,1.69,61,6.3,E


In [49]:
# Посмотрим информацию о новом датафрейме
df_actual.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12780 entries, 0 to 16954
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12780 non-null  object 
 1   platform         12780 non-null  object 
 2   year_of_release  12780 non-null  Int16  
 3   genre            12780 non-null  object 
 4   na_sales         12780 non-null  float64
 5   eu_sales         12780 non-null  float64
 6   jp_sales         12780 non-null  float64
 7   other_sales      12780 non-null  float64
 8   critic_score     12780 non-null  Int8   
 9   user_score       12780 non-null  float64
 10  rating           12780 non-null  object 
dtypes: Int16(1), Int8(1), float64(5), object(4)
memory usage: 1.0+ MB


<font color='#777778'>Получили новый датафрейм `df_actual` состоящий из 12780 строк и 11 столбцов. </font>

---

## 4. <a id='category'>Категоризация данных</a>

<font color='#777778'>Проведем категоризацию данных по оценкам пользователей. Выделим три категории:</font>

<font color='#777778'>1. **высокая оценка** (от 8 до 10 включительно)</font>

<font color='#777778'>2. **средняя оценка** (от 3 до 8, не включая правую границу интервала)</font>

<font color='#777778'>3. **низкая оценка** (от 0 до 3, не включая правую границу интервала)</font>

In [50]:
# Для определения категории создадим новый столбец user_category и воспользуемся функцией cut
df_actual.loc[:,'user_category'] = pd.cut(df_actual['user_score'], bins=[0,3,8,10], labels = ['низкая оценка','средняя оценка','высокая оценка'], right = False)

In [51]:
# Проверим правильно ли данные разбились на категории
df1 = df_actual[df_actual['user_category'].notna()]
df1.groupby('user_category', observed = True)['user_score'].agg(['min','max'])

,min,max
user_category,,
низкая оценка,0.0,2.9
средняя оценка,3.0,7.9
высокая оценка,8.0,9.7


<font color='#777778'>Разделим все игры по оценкам критиков и выделим такие категории:</font>

<font color='#777778'>1. **высокая оценка** (от 80 до 100 включительно) </font>

<font color='#777778'>2. **средняя оценка** (от 30 до 80, не включая правую границу интервала)</font>

<font color='#777778'>3. **низкая оценка** (от 0 до 30, не включая правую границу интервала).</font>

In [52]:
# Для определения категории создадим новый столбец critic_category и воспользуемся функцией cut
df_actual.loc[:,'critic_category'] = pd.cut(df_actual['critic_score'], bins=[0,30,80,100], labels = ['низкая оценка','средняя оценка','высокая оценка'], right = False)

In [53]:
# Проверим правильно ли данные разбились на категории
df2 = df_actual[df_actual['critic_category'].notna()]
df2.groupby('critic_category', observed = True)['critic_score'].agg(['min','max'])

,min,max
critic_category,,
низкая оценка,13,29
средняя оценка,30,79
высокая оценка,80,98


<font color='#777778'>Cгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории.</font>

In [54]:
# Посчитаем количество из в каждой категории по оценкам пользователей
df_actual.groupby('user_category', observed=True)['name'].count()

user_category
низкая оценка      116
средняя оценка    4080
высокая оценка    2286
Name: name, dtype: int64

In [55]:
# Посчитаем количество из в каждой категории по оценкам критиков
df_actual.groupby('critic_category', observed=True)['name'].count()

critic_category
низкая оценка       55
средняя оценка    5422
высокая оценка    1691
Name: name, dtype: int64

<font color='#777778'>По оценкам критиков и по оценкам пользователей преобладают категории с средней оценкой. 
Меньше всего количество игр с низкой оценкой.</font>

<font color='#777778'>Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.</font>

In [56]:
df_actual.groupby('platform')['name'].count().sort_values(ascending=False).head(7)

platform
PS2     2127
DS      2120
WII     1275
PSP     1180
X360    1121
PS3     1086
GBA      811
Name: name, dtype: int64

## 5. <a id='itog'>Итоговый вывод</a>

<font color='#777778'>Были загружены данные из датасета `/datasets/new_games.csv`. В результате получился датафрейм, состоящий из 16956 строк с 11 столбцов.
Далее названия столбцов были приведены к стилю `snake_case`. Было произведено преобразование столбцов к оптимальным типам данных, что позволило освободить память в размере 182,14 Кб. Была проиведена работа с пропусками в данных (пропуски в столбцах `name`, `genre` и `year_of_release` были удалены из датафрейма, пропуски в столбцах `eu_sales` и `jp_sales` были заменены средними значениями в зависимости от и года выхода игры, а пропуски в столбцах `critic_score`, `user_score` и `rating` были заменены значениями-индикаторами. Были найдены и удалены явные _(512 строк)_ и неявные дубликаты _(1 строка)_, в результате датафрейм сократился _на 513 строк (3,03%)_. Была произведена фильтрация датафрейма по году выхода игр. В него попали игры, вышедшие в период с 2000 по 2013гг. После чего появился новый датафрейм состоящий из  _12780_ строк и 11 столбцов. Далее была произведена категоризация по оценкам пользователей и критиков с категориями `низкая оценка`, `средняя оценка` и  `высокая оценка`. В датафрейме появились новые столбцы `user_category` и `critic_category`. Далее мы выделили топ-7 платформ по количеству выпущенных игр.</font>